# 🎙️ Ditto Audio Feature Extractor

This notebook **extracts HuBERT audio features** from your input WAV file using the exact same pipeline as `inference.py`.

The extracted features are `(N_frames, 1024)` float32 numpy arrays — the same format fed into the Ditto diffusion model (Audio2Motion / LMDM).

---
### Pipeline:
```
audio.wav  ─►  librosa (resample to 16kHz)  ─►  HuBERT ONNX (streaming)  ─►  features.npy  (N, 1024)
```
---

## Step 1 — Clone the Ditto repo & install dependencies

In [ ]:
# Clone repo (skip if already done)
import os

if not os.path.exists('ditto-talkinghead'):
    !git clone https://github.com/antgroup/ditto-talkinghead.git

os.chdir('ditto-talkinghead')
print('Working dir:', os.getcwd())

In [ ]:
# Install required packages
!pip install -q librosa onnxruntime-gpu numpy tqdm

# Verify GPU onnxruntime can see CUDA
import onnxruntime as ort
print('ONNX Runtime version:', ort.__version__)
print('Available providers:', ort.get_available_providers())

## Step 2 — Set the HuBERT ONNX model path

The checkpoint lives inside `checkpoints/ditto_pytorch/`.  
Adjust the path below if you store it elsewhere (e.g. Google Drive).

In [ ]:
import os
from huggingface_hub import hf_hub_download

# Define target directory
base_dir = "./mimi-to-hubert-bridge"
model_dir = os.path.join(base_dir, "model")

# Create directories if they don't exist
os.makedirs(model_dir, exist_ok=True)

# Download the file into the model folder
file_path = hf_hub_download(
    repo_id="digital-avatar/ditto-talkinghead",
    filename="ditto_pytorch/aux_models/hubert_streaming_fix_kv.onnx",
    local_dir=model_dir,
    local_dir_use_symlinks=False
)

print("Downloaded to:", file_path)

In [ ]:
# ── Mount Google Drive if your checkpoints are there ──────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')

# ── Set HuBERT ONNX model path ─────────────────────────────────────────────────
# Option A: in repo checkpoints (default)
HUBERT_MODEL_PATH = './mimi-to-hubert-bridge/model/ditto_pytorch/aux_models/hubert_streaming_fix_kv.onnx'

# Option B: from Google Drive
# HUBERT_MODEL_PATH = '/content/drive/MyDrive/ditto_checkpoints/hubert_streaming_fix_kv.onnx'

import os
assert os.path.isfile(HUBERT_MODEL_PATH), f'HuBERT model not found: {HUBERT_MODEL_PATH}'
print('✅ HuBERT model found:', HUBERT_MODEL_PATH)

## Step 3 — Upload your audio file

In [ ]:
from google.colab import files

print('Upload your audio file (WAV, MP3, FLAC, etc.):')
uploaded = files.upload()

# Use the first uploaded file
AUDIO_PATH = list(uploaded.keys())[0]
print(f'\n✅ Audio uploaded: {AUDIO_PATH}')

## Step 4 — Audio Feature Extraction (self-contained, no Ditto package needed)

This is a **standalone copy** of:
- `core/aux_models/modules/hubert_stream.py` → `HubertStreamingONNX`  
- `core/atomic_components/wav2feat.py` → `Wav2FeatHubert.wav2feat()`

**Bug fix**: The ONNX model `hubert_streaming_fix_kv.onnx` returns shape `(T', 1024)` directly (no batch dim).  
We now check `ndim` at runtime and handle both `(T', 1024)` and `(1, T', 1024)` safely.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Standalone re-implementation of Ditto's audio feature extraction
# Identical logic to:
#   core/aux_models/modules/hubert_stream.py  →  HubertStreamingONNX
#   core/atomic_components/wav2feat.py        →  Wav2FeatHubert.wav2feat()
# ─────────────────────────────────────────────────────────────────────────────

import math
import numpy as np
import librosa
import onnxruntime
from tqdm import tqdm


# ── 1. HuBERT ONNX inference wrapper ─────────────────────────────────────────
class HubertStreamingONNX:
    """
    Mirrors: core/aux_models/modules/hubert_stream.py :: HubertStreamingONNX

    Runs the HuBERT ONNX model chunk-by-chunk.
    Input  : audio_chunk  float32 [1, T]  (T = split_len samples at 16 kHz)
    Output : encoding     float32 (T', 1024)  — batch dim auto-stripped

    NOTE: hubert_streaming_fix_kv.onnx outputs (T', 1024) directly (no batch
          dim). We handle both (T', 1024) and (1, T', 1024) shapes safely.
    """

    def __init__(self, model_file: str, device: str = 'cuda'):
        if device == 'cuda':
            providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        else:
            providers = ['CPUExecutionProvider']

        self.session = onnxruntime.InferenceSession(model_file, providers=providers)
        print(f'[HubertStreamingONNX] loaded   : {model_file}')
        print(f'[HubertStreamingONNX] providers: {self.session.get_providers()}')

        # Log actual ONNX output spec so shape is always visible
        out = self.session.get_outputs()[0]
        print(f'[HubertStreamingONNX] output   : name={out.name}  shape={out.shape}  type={out.type}')

    def forward_chunk(self, audio_chunk: np.ndarray) -> np.ndarray:
        """
        audio_chunk : 1-D float32 array, length = split_len (e.g. 6480 samples)
        returns     : 2-D float32 array, shape (T', 1024)

        The ONNX model may return:
          • (T', 1024)    — no batch dim  (hubert_streaming_fix_kv.onnx)
          • (1, T', 1024) — with batch dim
        We normalise to (T', 1024) in both cases.
        """
        raw = self.session.run(
            None,
            {'input_values': audio_chunk.reshape(1, -1).astype(np.float32)}
        )[0]                         # (T', 1024)  OR  (1, T', 1024)

        if raw.ndim == 3:            # (1, T', 1024) → (T', 1024)
            raw = raw[0]
        # raw.ndim == 2: already (T', 1024), nothing to do
        return raw                   # shape: (T', 1024)

    def __call__(self, audio_chunk: np.ndarray) -> np.ndarray:
        return self.forward_chunk(audio_chunk)


# ── 2. Full offline wav → features pipeline ───────────────────────────────────
class Wav2FeatHubert:
    """
    Mirrors: core/atomic_components/wav2feat.py :: Wav2FeatHubert

    Takes a raw waveform and returns HuBERT features:
        output shape : (N_frames, 1024)   float32
    where N_frames ≈ audio_duration_seconds × 25  (25 FPS)
    """

    # Default chunk params — must match what the ONNX model was exported with
    DEFAULT_CHUNKSIZE = (3, 5, 2)   # (pre_frames, valid_frames, post_frames)

    def __init__(self, model_path: str, device: str = 'cuda'):
        self.hubert = HubertStreamingONNX(model_path, device=device)

    def _forward_chunk(self, audio_chunk: np.ndarray,
                       chunksize: tuple = DEFAULT_CHUNKSIZE) -> np.ndarray:
        """
        Process one audio chunk → valid HuBERT frames.
        Returns: (chunksize[1], 1024)  e.g. (5, 1024)

        Mirrors Wav2FeatHubert.__call__() in wav2feat.py
        """
        # Slice indices into the per-chunk encoding
        # chunksize=(3,5,2):
        #   valid_feat_s = -(5+2)*2 = -14
        #   valid_feat_e = -2*2    = -4
        #   → 10 rows of shape (1024,) → reshape to (5, 2, 1024) → mean over axis 1
        valid_feat_s = -sum(chunksize[1:]) * 2    # e.g. -14
        valid_feat_e = -chunksize[2] * 2           # e.g.  -4

        encoding_chunk = self.hubert(audio_chunk)              # (T', 1024)
        valid_encoding = encoding_chunk[valid_feat_s:valid_feat_e]  # (10, 1024)

        # Average every pair of consecutive HuBERT frames → 25 FPS
        valid_feat = valid_encoding.reshape(chunksize[1], 2, 1024).mean(1)  # (5, 1024)
        return valid_feat

    def wav2feat(self, audio: np.ndarray, sr: int = 16000,
                 chunksize: tuple = DEFAULT_CHUNKSIZE) -> np.ndarray:
        """
        Full offline extraction. Mirrors Wav2FeatHubert.wav2feat() exactly.

        Args:
            audio     : raw waveform (any sample rate)
            sr        : sample rate of `audio`
            chunksize : (pre, valid, post) frame counts

        Returns:
            np.ndarray  shape (N_frames, 1024)  dtype float32
        """
        # 1. Resample to 16 kHz
        if sr != 16000:
            print(f'  Resampling {sr} Hz → 16000 Hz ...')
            audio_16k = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        else:
            audio_16k = audio

        # 2. Total output frames at 25 FPS
        num_f = math.ceil(len(audio_16k) / 16000 * 25)

        # 3. Chunk size in samples
        #    = total_frames × 40 ms × 16000 Hz + 80 (HuBERT conv offset)
        split_len = int(sum(chunksize) * 0.04 * 16000) + 80    # e.g. 6480

        # 4. Pad: prepend silence to align first valid chunk
        pre_silence = split_len - int(sum(chunksize[1:]) * 0.04 * 16000)  # 2160
        speech_pad = np.concatenate([
            np.zeros((pre_silence,), dtype=audio_16k.dtype),
            audio_16k,
            np.zeros((split_len,), dtype=audio_16k.dtype),
        ], axis=0)

        # 5. Slide and extract
        res_lst = []
        i = 0
        pbar = tqdm(total=math.ceil(num_f / chunksize[1]),
                    desc='Extracting HuBERT features')
        while i < num_f:
            sss = int(i * 0.04 * 16000)
            eee = sss + split_len
            chunk = speech_pad[sss:eee].astype(np.float32)
            valid_feat = self._forward_chunk(chunk, chunksize)  # (valid_frames, 1024)
            res_lst.append(valid_feat)
            i += chunksize[1]   # advance by valid_frames
            pbar.update()
        pbar.close()

        # 6. Concat and trim to exact frame count
        ret = np.concatenate(res_lst, axis=0)   # (N_padded, 1024)
        ret = ret[:num_f]                        # (N_frames, 1024)
        return ret


print('✅ Extractor classes defined.')

## Step 5 — Run Extraction

In [ ]:
import librosa
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE         = 'cuda'               # 'cuda' or 'cpu'
OUTPUT_NPY     = 'audio_features.npy' # output filename
CHUNKSIZE      = (3, 5, 2)            # DO NOT CHANGE — must match the ONNX model

# ── Load audio (same as inference.py line 36) ─────────────────────────────────
print(f'Loading audio: {AUDIO_PATH}')
audio, sr = librosa.core.load(AUDIO_PATH, sr=16000)
duration = len(audio) / sr
expected_frames = math.ceil(len(audio) / 16000 * 25)
print(f'  Sample rate : {sr} Hz')
print(f'  Duration    : {duration:.2f} s')
print(f'  Expected    : {expected_frames} frames @ 25 FPS')

# ── Build extractor ───────────────────────────────────────────────────────────
print('\nLoading HuBERT ONNX model ...')
extractor = Wav2FeatHubert(model_path=HUBERT_MODEL_PATH, device=DEVICE)

# ── Extract features ──────────────────────────────────────────────────────────
print('\nRunning feature extraction ...')
features = extractor.wav2feat(audio, sr=16000, chunksize=CHUNKSIZE)

# ── Report ────────────────────────────────────────────────────────────────────
print(f'\n✅ Features extracted!')
print(f'   Shape : {features.shape}  →  (N_frames, 1024)')
print(f'   dtype : {features.dtype}')
print(f'   min   : {features.min():.4f}  |  max : {features.max():.4f}')

# ── Save ──────────────────────────────────────────────────────────────────────
np.save(OUTPUT_NPY, features)
print(f'\n💾 Saved to: {OUTPUT_NPY}')

## Step 6 — Visualise & verify

In [ ]:
import matplotlib
matplotlib.use('Agg')  # headless-safe
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Feature heatmap
axes[0].imshow(features.T, aspect='auto', origin='lower',
               interpolation='nearest', cmap='viridis')
axes[0].set_xlabel('Frame index (25 FPS)')
axes[0].set_ylabel('Feature dimension (1024)')
axes[0].set_title(f'HuBERT Features  {features.shape}')
plt.colorbar(axes[0].images[0], ax=axes[0])

# Per-frame L2 norm
norms = np.linalg.norm(features, axis=1)
axes[1].plot(norms)
axes[1].set_xlabel('Frame index')
axes[1].set_ylabel('L2 norm')
axes[1].set_title('Per-frame L2 norm of HuBERT features')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('features_plot.png', dpi=100)
plt.show()
print('Plot saved to features_plot.png')

## Step 7 — Download the features

In [ ]:
from google.colab import files
files.download(OUTPUT_NPY)
print(f'Downloaded: {OUTPUT_NPY}')

---
## 📌 Notes — How to feed features into the diffusion model

These features are **exactly** what `inference.py` puts into `SDK.audio2motion_queue`:

```python
# inference.py (offline mode)
aud_feat = SDK.wav2feat.wav2feat(audio)   # ← what we replicate above
SDK.audio2motion_queue.put(aud_feat)      # ← feed into LMDM diffusion model
```

Loading saved features back:

```python
import numpy as np
features = np.load('audio_features.npy')   # shape (N, 1024), float32
SDK.audio2motion_queue.put(features)
```

### Format contract
| Property | Value |
|---|---|
| Shape | `(N_frames, 1024)` |
| dtype | `float32` |
| Frame rate | 25 FPS (one feature per video frame) |
| Encoder | HuBERT streaming (ONNX) |
| Aggregation | 2 HuBERT frames averaged → 1 output frame |

### What was fixed (v2)
The original notebook assumed the ONNX output had a leading batch dim `(1, T', 1024)` and indexed it with `[0]`.  
`hubert_streaming_fix_kv.onnx` returns `(T', 1024)` directly — indexing `[0]` gave a single 1024-d row instead of the full (T', 1024) tensor, causing the later reshape to fail.  
Now we inspect `ndim` at runtime and handle both cases correctly.